# Collaborative filtering

## Imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

## Load data

In [2]:
PATH = Path("./../ebnerd_demo")
data_split = "train"

In [3]:
df_behaviors = pd.read_parquet(PATH.joinpath(data_split, "behaviors.parquet"))
df_history = pd.read_parquet(PATH.joinpath(data_split, "history.parquet"))
df_articles = pd.read_parquet(PATH.joinpath("articles.parquet"))

In [4]:
df_behaviors.head(5)

,impression_id,article_id,impression_time,read_time,scroll_percentage,device_type,article_ids_inview,article_ids_clicked,user_id,is_sso_user,gender,postcode,age,is_subscriber,session_id,next_read_time,next_scroll_percentage
0,48401,NaN,2023-05-21 21:06:50,21.0,NaN,2,"[9774516, 9771051, 9770028, 9775402, 9774461, ...",[9759966],22779,False,NaN,NaN,NaN,False,21,16.0,27.0
1,152513,9778745.0,2023-05-24 07:31:26,30.0,100.0,1,"[9778669, 9778736, 9778623, 9089120, 9778661, ...",[9778661],150224,False,NaN,NaN,NaN,False,298,2.0,48.0
2,155390,NaN,2023-05-24 07:30:33,45.0,NaN,1,"[9778369, 9777856, 9778500, 9778021, 9778627, ...",[9777856],160892,False,NaN,NaN,NaN,False,401,215.0,100.0
3,214679,NaN,2023-05-23 05:25:40,33.0,NaN,2,"[9776715, 9776406, 9776566, 9776071, 9776808, ...",[9776566],1001055,False,NaN,NaN,NaN,False,1357,40.0,47.0
4,214681,NaN,2023-05-23 05:31:54,21.0,NaN,2,"[9775202, 9776855, 9776688, 9771995, 9776583, ...",[9776553],1001055,False,NaN,NaN,NaN,False,1358,5.0,49.0


In [5]:
df_history.head(5)

,user_id,impression_time_fixed,scroll_percentage_fixed,article_id_fixed,read_time_fixed
0,13538,"[2023-04-27T10:17:43.000000, 2023-04-27T10:18:...","[100.0, 35.0, 100.0, 24.0, 100.0, 23.0, 100.0,...","[9738663, 9738569, 9738663, 9738490, 9738663, ...","[17.0, 12.0, 4.0, 5.0, 4.0, 9.0, 5.0, 46.0, 11..."
1,58608,"[2023-04-27T18:48:09.000000, 2023-04-27T18:48:...","[37.0, 61.0, 100.0, 100.0, 55.0, 100.0, 100.0,...","[9739362, 9739179, 9738567, 9739344, 9739202, ...","[2.0, 24.0, 72.0, 65.0, 11.0, 4.0, 101.0, 0.0,..."
2,95507,"[2023-04-27T15:20:28.000000, 2023-04-27T15:20:...","[60.0, 100.0, 100.0, 21.0, 29.0, 67.0, 49.0, 5...","[9739035, 9738646, 9634967, 9738902, 9735495, ...","[18.0, 29.0, 51.0, 12.0, 10.0, 10.0, 13.0, 24...."
3,106588,"[2023-04-27T08:29:09.000000, 2023-04-27T08:29:...","[24.0, 57.0, 100.0, nan, nan, 100.0, 100.0, 73...","[9738292, 9738216, 9737266, 9737556, 9737657, ...","[9.0, 15.0, 42.0, 9.0, 3.0, 58.0, 26.0, 214.0,..."
4,617963,"[2023-04-27T14:42:25.000000, 2023-04-27T14:43:...","[100.0, 100.0, nan, 46.0, 23.0, 19.0, 61.0, 70...","[9739035, 9739088, 9738902, 9738968, 9738760, ...","[45.0, 29.0, 116.0, 26.0, 34.0, 42.0, 58.0, 59..."


In [6]:
df_articles.head(5)

,article_id,title,subtitle,last_modified_time,premium,body,published_time,image_ids,article_type,url,...,entity_groups,topics,category,subcategory,category_str,total_inviews,total_pageviews,total_read_time,sentiment_score,sentiment_label
0,3037230,Ishockey-spiller: Jeg troede jeg skulle dø,ISHOCKEY: Ishockey-spilleren Sebastian Harts h...,2023-06-29 06:20:57,False,Ambitionerne om at komme til USA og spille ish...,2003-08-28 08:55:00,None,article_default,https://ekstrabladet.dk/sport/anden_sport/isho...,...,[],"[Kriminalitet, Kendt, Sport, Katastrofe, Mindr...",142,"[327, 334]",sport,NaN,NaN,NaN,0.9752,Negative
1,3044020,Prins Harry tvunget til dna-test,Hoffet tvang Prins Harry til at tage dna-test ...,2023-06-29 06:21:16,False,Den britiske tabloidavis The Sun fortsætter me...,2005-06-29 08:47:00,"[3097307, 3097197, 3104927]",article_default,https://ekstrabladet.dk/underholdning/udlandke...,...,"[PER, PER]","[Kriminalitet, Kendt, Underholdning, Personfar...",414,[432],underholdning,NaN,NaN,NaN,0.7084,Negative
2,3057622,Rådden kørsel på blå plader,Kan ikke straffes: Udenlandske diplomater i Da...,2023-06-29 06:21:24,False,Slingrende spritkørsel. Grove overtrædelser af...,2005-10-10 07:20:00,[3047102],article_default,https://ekstrabladet.dk/nyheder/samfund/articl...,...,[],"[Kriminalitet, Transportmiddel, Bil]",118,[133],nyheder,NaN,NaN,NaN,0.9236,Negative
3,3073151,Mærsk-arvinger i livsfare,FANGET I FLODBØLGEN: Skibsrederens oldebørn må...,2023-06-29 06:21:38,False,To oldebørn af skibsreder Mærsk McKinney Mølle...,2005-01-04 06:59:00,"[3067474, 3067478, 3153705]",article_default,https://ekstrabladet.dk/nyheder/samfund/articl...,...,[],"[Erhverv, Privat virksomhed, Livsstil, Familie...",118,[133],nyheder,NaN,NaN,NaN,0.9945,Negative
4,3193383,Skød svigersøn gennem babydyne,44-årig kvinde tiltalt for drab på ekssvigersø...,2023-06-29 06:22:57,False,En 44-årig mormor blev i dag fremstillet i et ...,2003-09-15 15:30:00,None,article_default,https://ekstrabladet.dk/krimi/article3193383.ece,...,[],"[Kriminalitet, Personfarlig kriminalitet]",140,[],krimi,NaN,NaN,NaN,0.9966,Negative


In [7]:
df = pd.merge(df_behaviors,df_history, how='left', on='user_id')
df.head()

,impression_id,article_id,impression_time,read_time,scroll_percentage,device_type,article_ids_inview,article_ids_clicked,user_id,is_sso_user,...,postcode,age,is_subscriber,session_id,next_read_time,next_scroll_percentage,impression_time_fixed,scroll_percentage_fixed,article_id_fixed,read_time_fixed
0,48401,NaN,2023-05-21 21:06:50,21.0,NaN,2,"[9774516, 9771051, 9770028, 9775402, 9774461, ...",[9759966],22779,False,...,NaN,NaN,False,21,16.0,27.0,"[2023-04-27T09:05:54.000000, 2023-04-27T09:06:...","[28.0, 17.0, 35.0, 4.0, 77.0, 26.0, 53.0, 59.0...","[9738452, 9737521, 9738760, 9733713, 9738777, ...","[5.0, 4.0, 3.0, 21.0, 18.0, 4.0, 23.0, 3.0, 6...."
1,152513,9778745.0,2023-05-24 07:31:26,30.0,100.0,1,"[9778669, 9778736, 9778623, 9089120, 9778661, ...",[9778661],150224,False,...,NaN,NaN,False,298,2.0,48.0,"[2023-04-29T11:34:06.000000, 2023-04-29T11:34:...","[100.0, 49.0, 100.0, 100.0, 72.0, 100.0, 85.0,...","[9740087, 9741986, 9740591, 9741830, 9742491, ...","[18.0, 244.0, 32.0, 36.0, 31.0, 58.0, 65.0, 59..."
2,155390,NaN,2023-05-24 07:30:33,45.0,NaN,1,"[9778369, 9777856, 9778500, 9778021, 9778627, ...",[9777856],160892,False,...,NaN,NaN,False,401,215.0,100.0,"[2023-04-27T09:10:33.000000, 2023-04-27T09:20:...","[100.0, 20.0, 100.0, 100.0, 100.0, 15.0, 58.0,...","[9738557, 9738211, 9736646, 9739342, 9739443, ...","[583.0, 257.0, 10.0, 396.0, 28.0, 6.0, 3.0, 10..."
3,214679,NaN,2023-05-23 05:25:40,33.0,NaN,2,"[9776715, 9776406, 9776566, 9776071, 9776808, ...",[9776566],1001055,False,...,NaN,NaN,False,1357,40.0,47.0,"[2023-04-27T12:12:45.000000, 2023-04-27T12:13:...","[100.0, 100.0, 39.0, 100.0, 100.0, 97.0, 35.0,...","[9738777, 9738663, 9736693, 9738656, 9736689, ...","[35.0, 62.0, 6.0, 48.0, 71.0, 95.0, 6.0, 6.0, ..."
4,214681,NaN,2023-05-23 05:31:54,21.0,NaN,2,"[9775202, 9776855, 9776688, 9771995, 9776583, ...",[9776553],1001055,False,...,NaN,NaN,False,1358,5.0,49.0,"[2023-04-27T12:12:45.000000, 2023-04-27T12:13:...","[100.0, 100.0, 39.0, 100.0, 100.0, 97.0, 35.0,...","[9738777, 9738663, 9736693, 9738656, 9736689, ...","[35.0, 62.0, 6.0, 48.0, 71.0, 95.0, 6.0, 6.0, ..."


In [8]:
df_behaviors.columns

Index(['impression_id', 'article_id', 'impression_time', 'read_time',
       'scroll_percentage', 'device_type', 'article_ids_inview',
       'article_ids_clicked', 'user_id', 'is_sso_user', 'gender', 'postcode',
       'age', 'is_subscriber', 'session_id', 'next_read_time',
       'next_scroll_percentage'],
      dtype='object')

In [9]:
df_articles.columns

Index(['article_id', 'title', 'subtitle', 'last_modified_time', 'premium',
       'body', 'published_time', 'image_ids', 'article_type', 'url',
       'ner_clusters', 'entity_groups', 'topics', 'category', 'subcategory',
       'category_str', 'total_inviews', 'total_pageviews', 'total_read_time',
       'sentiment_score', 'sentiment_label'],
      dtype='object')

In [10]:
df_history.columns

Index(['user_id', 'impression_time_fixed', 'scroll_percentage_fixed',
       'article_id_fixed', 'read_time_fixed'],
      dtype='object')

In [11]:
df.columns

Index(['impression_id', 'article_id', 'impression_time', 'read_time',
       'scroll_percentage', 'device_type', 'article_ids_inview',
       'article_ids_clicked', 'user_id', 'is_sso_user', 'gender', 'postcode',
       'age', 'is_subscriber', 'session_id', 'next_read_time',
       'next_scroll_percentage', 'impression_time_fixed',
       'scroll_percentage_fixed', 'article_id_fixed', 'read_time_fixed'],
      dtype='object')

## Encode columns

In [12]:
df["impression_time"] = pd.to_datetime(df["impression_time"])
df["hour"] = df["impression_time"].dt.hour
df["day_of_week"] = df["impression_time"].dt.dayofweek
df["article_ids_inview"]

0        [9774516, 9771051, 9770028, 9775402, 9774461, ...
1        [9778669, 9778736, 9778623, 9089120, 9778661, ...
2        [9778369, 9777856, 9778500, 9778021, 9778627, ...
3        [9776715, 9776406, 9776566, 9776071, 9776808, ...
4        [9775202, 9776855, 9776688, 9771995, 9776583, ...
                               ...                        
24719    [9775562, 9775701, 9486486, 9353420, 9775621, ...
24720    [9775347, 9678300, 9754160, 9770028, 9775500, ...
24721        [9775699, 9775722, 9774541, 9775648, 9775325]
24722    [9772221, 9758424, 9775699, 9775722, 9775621, ...
24723    [9657528, 9769356, 9746428, 9771237, 9770369, ...
Name: article_ids_inview, Length: 24724, dtype: object

In [13]:
df_selected = df[["article_ids_clicked", "article_ids_inview"]]

def create_binary_labels_column(df, clicked_col, inview_col, shuffle=False, seed=None):
    df_copy = df.copy()
    df_copy['labels'] = df_copy.apply(lambda row: [1 if article in row[clicked_col] else 0 for article in row[inview_col]], axis=1)
    
    if shuffle:
        df_copy = df_copy.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    return df_copy

df_labeled = create_binary_labels_column(
    df_selected, "article_ids_clicked", "article_ids_inview", shuffle=True, seed=123
)

df_labeled["labels_len"] = df_labeled["labels"].apply(len)

In [14]:
df_labeled

,article_ids_clicked,article_ids_inview,labels,labels_len
0,[9779541],"[9518688, 9778788, 9779541, 9052240, 9115412, ...","[0, 0, 1, 0, 0, 0]",6
1,[9779489],"[9779674, 9777321, 9779641, 9779489, 9779517]","[0, 0, 0, 1, 0]",5
2,[9777182],"[9779377, 9779507, 9777182, 9776474, 9777397, ...","[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",13
3,[9769762],"[9770886, 9771065, 9771126, 9769762, 9771166, ...","[0, 0, 0, 1, 0, 0]",6
4,[9772475],"[9772300, 9773210, 9772475, 9773282, 9772601, ...","[0, 0, 1, 0, 0, 0]",6
...,...,...,...,...
24719,[9770051],"[7213923, 9432542, 9771627, 9770051, 9099237, ...","[0, 0, 0, 1, 0, 0]",6
24720,[9771627],"[9099237, 9364014, 9516530, 9407487, 9771627]","[0, 0, 0, 0, 1]",5
24721,[9777005],"[9777005, 9777036, 9775908, 9777075, 9777043, ...","[1, 0, 0, 0, 0, 0, 0, 0]",8
24722,[9774297],"[9774297, 9773857, 9771996, 8514670, 9539706]","[1, 0, 0, 0, 0]",5


## Interaction Matrix

In [15]:
from sklearn.preprocessing import MinMaxScaler

In [16]:
df["interaction"] = 1
interaction_df = (
    df.groupby(["user_id", "article_id"])[
        ["interaction", "read_time", "scroll_percentage"]
    ]
    .agg(list)
    .reset_index()
)
interaction_df["interaction"] = interaction_df["interaction"].apply(lambda x: sum(x))
interaction_df["read_time"] = interaction_df["read_time"].apply(lambda x: sum(x))
interaction_df["scroll_percentage"] = interaction_df["scroll_percentage"].apply(
    lambda x: max(x)
)
scaler = MinMaxScaler()

# interaction_df[["interaction","read_time","scroll_percentage"]] = pd.DataFrame(scaler.fit_transform(interaction_df[["interaction","read_time","scroll_percentage"]]), columns=["interaction","read_time","scroll_percentage"])

In [17]:
interaction_df

,user_id,article_id,interaction,read_time,scroll_percentage
0,11313,9775573.0,1,13.0,100.0
1,11313,9778682.0,1,51.0,100.0
2,19892,9735909.0,1,7.0,100.0
3,21271,9776449.0,1,21.0,100.0
4,22779,9771367.0,1,64.0,100.0
...,...,...,...,...,...
5642,2585449,9777750.0,1,89.0,100.0
5643,2585449,9777822.0,2,56.0,100.0
5644,2585449,9778769.0,1,670.0,100.0
5645,2585449,9779489.0,1,184.0,100.0


In [18]:
interaction_df["article_id"] = interaction_df["article_id"].astype(int)
interaction_matrix = interaction_df.pivot(
    index="user_id", columns="article_id", values="interaction"
).fillna(0)
interaction_matrix = interaction_matrix.sort_index(axis=0).sort_index(axis=1)
interaction_sparse_matrix = csr_matrix(interaction_matrix.values)

In [19]:
item_ids = interaction_df['article_id'].unique()
item_ids.sort()
user_ids = interaction_df['user_id'].unique()
user_ids.sort()

In [20]:
def get_index_by_value(array: np.ndarray, value):
    result = np.where(array == value)
    # Convert the result to a list of tuples
    indices = list(zip(*result))
    # Return the first index if it exists, else return None
    if indices:
        return indices[0]
    else:
        return None

In [21]:
interaction_matrix

article_id,3971783,5563498,6566219,6835130,7306652,7382590,8055313,8315213,8462836,8918299,...,9779956,9779968,9780019,9780020,9780039,9780096,9780181,9780193,9780195,9780271
user_id,,,,,,,,,,,,,,,,,,,,,
11313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19892,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
21271,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22779,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
22895,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2574644,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
2581162,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2583477,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
user_df = df.groupby(["user_id"]).agg(list)

In [23]:
user_df.head()

,impression_id,article_id,impression_time,read_time,scroll_percentage,device_type,article_ids_inview,article_ids_clicked,is_sso_user,gender,...,session_id,next_read_time,next_scroll_percentage,impression_time_fixed,scroll_percentage_fixed,article_id_fixed,read_time_fixed,hour,day_of_week,interaction
user_id,,,,,,,,,,,,,,,,,,,,,
11313,"[285722079, 382939361, 467751038, 487155599, 4...","[nan, nan, 9778682.0, nan, nan, nan, nan, 9775...","[2023-05-23 08:11:22, 2023-05-25 06:28:25, 202...","[11.0, 21.0, 51.0, 4.0, 12.0, 9.0, 108.0, 13.0]","[nan, nan, 100.0, nan, nan, nan, nan, 100.0]","[1, 1, 1, 1, 1, 1, 1, 1]","[[9776968, 9776917, 9777036, 9776897, 9777005,...","[[9776897], [9779860], [9778627], [9776855], [...","[False, False, False, False, False, False, Fal...","[nan, nan, nan, nan, nan, nan, nan, nan]",...,"[173619, 173629, 173622, 173618, 173617, 17362...","[0.0, 442.0, 7.0, 4.0, 708.0, 5.0, 6.0, 526.0]","[nan, 100.0, 73.0, 59.0, 58.0, 48.0, 68.0, 94.0]","[[2023-04-27T07:18:15.000000, 2023-04-27T07:35...","[[60.0, 100.0, 100.0, 1.0, 47.0, 100.0, 100.0,...","[[9737535, 9738334, 9738292, 9733713, 9740161,...","[[1038.0, 4.0, 32.0, 7.0, 4.0, 34.0, 640.0, 9....","[8, 6, 6, 6, 6, 7, 6, 6]","[1, 3, 2, 1, 1, 2, 0, 0]","[1, 1, 1, 1, 1, 1, 1, 1]"
13538,"[8245150, 42080531, 51830199, 164792472, 22069...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[2023-05-22 05:11:37, 2023-05-19 05:09:26, 202...","[11.0, 18.0, 12.0, 25.0, 8.0, 26.0, 28.0, 6.0,...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]","[[9754160, 9775485, 9768866, 9142581, 9770028,...","[[9754160], [9771187], [9773768], [9772343], [...","[False, False, False, False, False, False, Fal...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",...,"[74346, 74327, 74338, 74330, 74345, 74345, 743...","[6.0, 0.0, 2.0, 5.0, 6.0, 4.0, 26.0, 14.0, 0.0...","[9.0, nan, 19.0, 32.0, 46.0, 43.0, 33.0, 63.0,...","[[2023-04-27T10:17:43.000000, 2023-04-27T10:18...","[[100.0, 35.0, 100.0, 24.0, 100.0, 23.0, 100.0...","[[9738663, 9738569, 9738663, 9738490, 9738663,...","[[17.0, 12.0, 4.0, 5.0, 4.0, 9.0, 5.0, 46.0, 1...","[5, 5, 16, 20, 20, 20, 18, 12, 16, 1, 13, 13]","[0, 4, 5, 4, 6, 6, 4, 3, 3, 6, 6, 6]","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"
15430,"[52792944, 52792945, 453707725]","[nan, nan, nan]","[2023-05-24 03:17:37, 2023-05-24 03:17:22, 202...","[21.0, 12.0, 146.0]","[nan, nan, nan]","[1, 1, 1]","[[9142564, 9777307, 9777582, 9268227, 9777811,...","[[9761586], [9778375], [9770028]]","[False, False, False]","[nan, nan, nan]",...,"[173671, 173671, 173670]","[5.0, 2.0, 22.0]","[23.0, 60.0, 100.0]","[[2023-05-02T19:51:24.000000, 2023-05-02T19:53...","[[100.0, nan, 100.0, 100.0, 100.0, 73.0, 100.0...","[[9745909, 9746802, 9748576, 9748470, 9748576,...","[[101.0, 0.0, 32.0, 70.0, 12.0, 136.0, 75.0, 0...","[3, 3, 4]","[2, 2, 0]","[1, 1, 1]"
19181,"[52839103, 102389461, 107810664, 155654733, 16...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[2023-05-23 04:08:41, 2023-05-24 15:23:33, 202...","[22.0, 17.0, 53.0, 22.0, 12.0, 30.0, 12.0, 11....","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...","[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, ...","[[9776385, 9776046, 9776560, 9776566, 9776553,...","[[9776385], [9777636], [9774764], [9777475], [...","[False, False, False, False, False, False, Fal...","[nan, nan, nan, nan, nan, nan, nan, nan, nan, ...",...,"[173720, 173738, 173704, 173729, 173693, 17369...","[4.0, 3.0, 86.0, 195.0, 1.0, 5.0, 14.0, 21.0, ...","[40.0, 33.0, nan, 35.0, 20.0, 64.0, 100.0, 67....","[[2023-04-27T08:35:55.000000, 2023-04-27T08:36...","[[nan, 25.0, nan, 100.0, 100.0, 100.0, 21.0, 1...","[[9737521, 9738292, 9738216, 9737266, 9738182,...","[[2.0, 6.0, 1.0, 18.0, 24.0, 22.0, 4.0, 15.0, ...","[4, 15, 18, 14, 20, 20, 17, 4, 15, 17, 4, 4, 4...","[1, 2, 6, 1, 4, 4, 1, 5, 4, 0, 3, 2, 2, 2, 3, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
19568,"[53093781, 259085010]","[nan, nan]","[2023-05-25 06:24:25, 2023-05-1

## Recommender

In [24]:
user_item_matrix = interaction_df.pivot_table(index='user_id', columns='article_id', values='interaction', fill_value=0)
user_means = user_item_matrix.mean(axis=1)
user_item_matrix_normalized = user_item_matrix.sub(user_means, axis=0)

In [25]:
user_similarity = cosine_similarity(user_item_matrix_normalized)
user_similarity_df = pd.DataFrame(user_similarity, index=user_item_matrix.index, columns=user_item_matrix.index)

In [26]:
# Set article_id as index
df_articles.set_index('article_id', inplace=True)

In [27]:
df_articles['topics'] = df_articles['topics'].apply(lambda x: x if isinstance(x, list) else [])
df_articles['subcategory'] = df_articles['subcategory'].apply(lambda x: x if isinstance(x, list) else [])
df_articles['category'] = df_articles['category'].fillna('unknown')
df_articles[['total_pageviews', 'total_read_time', 'sentiment_score']] = df_articles[['total_pageviews', 'total_read_time', 'sentiment_score']].fillna(0)

# Convert lists to strings for TF-IDF and MultiLabelBinarizer
df_articles['topics'] = df_articles['topics'].apply(lambda x: ' '.join(x))




# Preprocess the features
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), ['category']),
        ('num', StandardScaler(), ['total_pageviews', 'total_read_time', 'sentiment_score']),
    ],
    remainder='drop'
)



# Fit and transform the data
metadata_normalized = preprocessor.fit_transform(df_articles[['topics','subcategory','category','total_pageviews', 'total_read_time', 'sentiment_score']])

# Calculate cosine similarity
item_similarity = cosine_similarity(metadata_normalized)
item_similarity_df = pd.DataFrame(item_similarity, index=df_articles.index, columns=df_articles.index)


In [28]:
def predict_interaction(user_id, article_id):
    if article_id not in user_item_matrix.columns:
        return "Article ID not in dataset"
    
    users_who_interacted = user_item_matrix[user_item_matrix[article_id] > 0].index

    # User-based CF
    numerator_user_cf = sum(user_similarity_df.loc[user_id, other_user] * user_item_matrix.loc[other_user, article_id]
                            for other_user in users_who_interacted)
    denominator_user_cf = sum(user_similarity_df.loc[user_id, other_user] for other_user in users_who_interacted)
    
    user_cf_prediction = 0
    if denominator_user_cf != 0:
        user_cf_prediction = numerator_user_cf / denominator_user_cf
    
    # Item-based CF
    interacted_items = user_item_matrix.columns[user_item_matrix.loc[user_id] > 0]
    numerator_item_cf = sum(item_similarity_df.loc[article_id, other_item] * user_item_matrix.loc[user_id, other_item]
                            for other_item in interacted_items)
    denominator_item_cf = sum(item_similarity_df.loc[article_id, other_item] for other_item in interacted_items)
    
    item_cf_prediction = 0
    if denominator_item_cf != 0:
        item_cf_prediction = numerator_item_cf / denominator_item_cf
    
    # Combine both predictions
    alpha = 0.5
    predicted_interaction = (alpha * user_cf_prediction) + ((1 - alpha) * item_cf_prediction)
    
    # Adjust with user mean
    predicted_interaction += user_means.loc[user_id]
    
    return predicted_interaction

print(predict_interaction(11313, 3971783))

1.002169197396963


In [29]:
def batch_predict(user, articles_inview):
    predictions = [
        (article, predict_interaction(user, article)) for article in articles_inview
    ]

    sorted_combined = sorted(predictions, key=lambda x: x[1])

    return [item[0] for item in sorted_combined]

## Test set

In [30]:
val_split = "validation"
df_behaviors_val = pd.read_parquet(PATH.joinpath(val_split, "behaviors.parquet"))
df_history_val = pd.read_parquet(PATH.joinpath(val_split, "history.parquet"))

In [31]:
df_behaviors.columns

Index(['impression_id', 'article_id', 'impression_time', 'read_time',
       'scroll_percentage', 'device_type', 'article_ids_inview',
       'article_ids_clicked', 'user_id', 'is_sso_user', 'gender', 'postcode',
       'age', 'is_subscriber', 'session_id', 'next_read_time',
       'next_scroll_percentage'],
      dtype='object')

In [32]:
df_history_val.columns

Index(['user_id', 'impression_time_fixed', 'scroll_percentage_fixed',
       'article_id_fixed', 'read_time_fixed'],
      dtype='object')

In [37]:
df_history

,user_id,impression_time_fixed,scroll_percentage_fixed,article_id_fixed,read_time_fixed
0,13538,"[2023-04-27T10:17:43.000000, 2023-04-27T10:18:...","[100.0, 35.0, 100.0, 24.0, 100.0, 23.0, 100.0,...","[9738663, 9738569, 9738663, 9738490, 9738663, ...","[17.0, 12.0, 4.0, 5.0, 4.0, 9.0, 5.0, 46.0, 11..."
1,58608,"[2023-04-27T18:48:09.000000, 2023-04-27T18:48:...","[37.0, 61.0, 100.0, 100.0, 55.0, 100.0, 100.0,...","[9739362, 9739179, 9738567, 9739344, 9739202, ...","[2.0, 24.0, 72.0, 65.0, 11.0, 4.0, 101.0, 0.0,..."
2,95507,"[2023-04-27T15:20:28.000000, 2023-04-27T15:20:...","[60.0, 100.0, 100.0, 21.0, 29.0, 67.0, 49.0, 5...","[9739035, 9738646, 9634967, 9738902, 9735495, ...","[18.0, 29.0, 51.0, 12.0, 10.0, 10.0, 13.0, 24...."
3,106588,"[2023-04-27T08:29:09.000000, 2023-04-27T08:29:...","[24.0, 57.0, 100.0, nan, nan, 100.0, 100.0, 73...","[9738292, 9738216, 9737266, 9737556, 9737657, ...","[9.0, 15.0, 42.0, 9.0, 3.0, 58.0, 26.0, 214.0,..."
4,617963,"[2023-04-27T14:42:25.000000, 2023-04-27T14:43:...","[100.0, 100.0, nan, 46.0, 23.0, 19.0, 61.0, 70...","[9739035, 9739088, 9738902, 9738968, 9738760, ...","[45.0, 29.0, 116.0, 26.0, 34.0, 42.0, 58.0, 59..."
...,...,...,...,...,...
1585,1122370,"[2023-05-09T06:50:19.000000, 2023-05-09T06:51:...","[100.0, 100.0, 100.0, 100.0, nan]","[9755181, 9706958, 9709329, 9706958, 9755571]","[84.0, 115.0, 298.0, 5.0, 79.0]"
1586,1718049,"[2023-05-17T05:04:36.000000, 2023-05-17T05:04:...","[53.0, 100.0, 24.0, 100.0, 83.0, 28.0, 37.0, 4...","[9768583, 9728166, 9768829, 9769641, 9769994, ...","[13.0, 17.0, 7.0, 99.0, 5.0, 8.0, 4.0, 2.0, 4.0]"
1587,1178033,"[2023-04-29T19:36:47.000000, 2023-04-29T19:37:...","[nan, 98.0, 92.0, 30.0, 35.0, 9.0, 34.0, 58.0,...","[9742268, 9742423, 9742270, 9742440, 9742459, ...","[10.0, 74.0, 89.0, 10.0, 1.0, 4.0, 11.0, 7.0, ..."
1588,395912,"[2023-05-12T05:12:34.000000, 2023-05-12T05:12:...","[100.0, 100.0, 100.0, 33.0, nan]","[9760741, 9760998, 9760935, 9754442, 9760935]","[16.0, 22.0, 23.0, 9.0, 0.0]"


In [43]:
x =df_history.explode(["article_id_fixed","scroll_percentage_fixed","read_time_fixed","impression_time_fixed"])

In [44]:
x["interaction"] = 1

In [49]:
interaction_df = x.groupby(["user_id","article_id_fixed"]).agg(list).reset_index()

In [50]:
interaction_df["interaction"] = interaction_df["interaction"].apply(
    lambda x: sum(x)
)
interaction_df["read_time_fixed"] = interaction_df["read_time_fixed"].apply(
    lambda x: sum(x)
)
interaction_df["scroll_percentage_fixed"] = interaction_df[
    "scroll_percentage_fixed"
].apply(lambda x: max(x))
interaction_df["score"] = (
    interaction_df["interaction"]
    + interaction_df["read_time_fixed"] / 120.0
    + interaction_df["scroll_percentage_fixed"] / 100.0
)

In [51]:
interaction_matrix = interaction_df.pivot(
    index="user_id", columns="article_id_fixed", values="score"
).fillna(0)